# 判別と地図化（team-alpha 版）— Phase 0

教材 `tutorials/05_result_map.ipynb` のチーム用コピー。
`04_field_pipeline_alpha.ipynb` が出した `field_features.csv` / `fields.geojson` を読み、
**Step4-5（3段階分類）・Step4-6（確信度と判定不能）**を適用して地図とファイルを出す。

## 教材からの主な変更点

1. **判定を機械学習からルールベースに置き換えた。** 教材は疑似ラベル→RandomForest→
   `predict_proba` のグレーゾーンを「要確認」とするが、本設計は採らない
   （`analysis_doc` Step4-6【設計判断】／`config.py` §6）
2. **クラス名を変えた。** 「休耕」→「**水稲作付が確認できない**」。真の休耕とは一致せず、
   転作・保全管理・耕作放棄の混合であるため（`analysis_doc`【判定と統計の対応関係】）
3. **「判定不能」を独立させた。** 観測不足の筆を3クラスのいずれにも入れない
4. **確信度を閾値からの距離で定義した。** `predict_proba` を使わない
5. 費用対効果の試算は Phase 0 では出さない（下記・最終セル）

## Phase 0 のスコープ

相馬市内 約7.2 km²・2024年・田のみ。**最小構成で解析計画が一気通貫で走るかの確認**であり、
範囲の代表性は要件ではない。ここで得た筆数・面積・要確認率は費用対効果に使わない。


In [ ]:
# 必要なライブラリのインストール（Colab / ローカルの Jupyter どちらでも同じ）
%pip -q install geopandas mapclassify japanize-matplotlib

In [ ]:
# ── ライブラリ読み込みと設定 ──
import os, sys, json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 110

try:
    import japanize_matplotlib
except ImportError:
    japanize_matplotlib = None

# ---- Colab なら Drive をマウント（ランタイムが切れるたびに必要）----
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive をマウントしました')

# ---- リポジトリを見つけて import パスを通す ----
import glob
_C = ['/content/drive/MyDrive/astrocamp-2026-sia/tutorials',
      os.path.join(os.getcwd(), 'tutorials'),
      os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', 'tutorials'))]
_C += [os.path.dirname(f) for f in
       glob.glob('/content/drive/MyDrive/*/tutorials/env.py') +
       glob.glob('/content/drive/MyDrive/*/*/tutorials/env.py')]
TUTORIALS_DIR = next((d for d in _C if os.path.exists(os.path.join(d, 'env.py'))), None)
if TUTORIALS_DIR is None:
    raise FileNotFoundError('env.py が見つかりません。探した場所:\n  ' + '\n  '.join(_C))
sys.path.insert(0, TUTORIALS_DIR)
REPO_DIR = os.path.dirname(TUTORIALS_DIR)
TEAM_DIR = os.path.join(REPO_DIR, 'submissions', 'camp', 'team-alpha')
sys.path.insert(0, TEAM_DIR)

from env import setup_font
setup_font()

# 【重要】config.py は必ずリロードする。
# Colab のランタイムは import 済みモジュールを sys.modules にキャッシュするため、
# git pull で config.py を更新してもランタイムを再起動しない限り古い値が返る
# （2026-09-08: N_OBS_MIN を 2 にしたのに None のまま扱われる事象が発生）。
import importlib
import config
importlib.reload(config)
cfg = config
print('config.py:', config.__file__)
print('repo:', REPO_DIR)
print('=== 再現性ヘッダ（config.py Version %s）===' % (
    getattr(cfg, '__doc__', None) or '1.2'))
print('ROI        :', cfg.ROI)
print('対象年     :', cfg.TARGET_YEAR)
print('湛水閾値   : 4月比', cfg.VH_FLOOD_DROP_DB, 'dB 以上の落ち込み')
print('生育閾値   : NDVI >=', cfg.NDVI_GROWTH_START)
print('N_OBS_MIN  :', cfg.N_OBS_MIN, '（None なら判定不能を分離できない）')
print('確信度の幅 :', cfg.CONFIDENCE_SCALE_DB, 'dB')

## 04 の成果物を読み込む

`field_features.csv`（1筆 × 1年 = 1行）と `fields.geojson`（筆ジオメトリ）。
**合成データへのフォールバックは置かない。** 教材は成果物が無いと合成データで続行するが、
提出物を作る場面で「実データのつもりが合成データだった」という取り違えは起こしてはならない。
無ければ止める。

In [ ]:
# ── 04 の成果物を読み込む ──
FCSV = os.path.join(TEAM_DIR, 'field_features.csv')
FGEO = os.path.join(TEAM_DIR, 'fields.geojson')
for p in (FCSV, FGEO):
    if not os.path.exists(p):
        raise FileNotFoundError(
            f'{p} がありません。先に 04_field_pipeline_alpha.ipynb を最後まで実行してください。')

field_features = pd.read_csv(FCSV)
with open(FGEO) as f:
    fields_gdf = gpd.GeoDataFrame.from_features(json.load(f)['features'], crs='EPSG:4326')
fields_gdf['field_id'] = fields_gdf['field_id'].astype(str)
field_features['field_id'] = field_features['field_id'].astype(str)

print('特徴量表  :', field_features.shape)
print('筆ポリゴン:', len(fields_gdf))
print('年        :', sorted(field_features['year'].unique()))

# 判定に必要な列がそろっているか（04 を古い版で回すと欠ける）
NEED = ['flood_detected', 'growth_detected', 'vh_drop_max', 'vh_apr', 'n_obs_min']
missing = [c for c in NEED if c not in field_features.columns]
if missing:
    raise KeyError('判定に必要な列がありません: %s\n'
                   '04_field_pipeline_alpha.ipynb を最新版で再実行してください。' % missing)
print('判定に必要な列: すべて存在')
field_features.head()

## `N_OBS_MIN` を決めるための分布

`config.py` の `N_OBS_MIN` は未定（`None`）。**この値が決まらないと「判定不能」を分離できない。**

`n_obs_min` は「4〜8月のうち、最も観測が少なかった月の採用観測数」。
下の分布を見て、どこで切るかをチームで決める。

In [ ]:
# ── n_obs の分布（N_OBS_MIN を決める材料）──
tgt = field_features[field_features['year'] == cfg.TARGET_YEAR]
if len(tgt) == 0:
    tgt = field_features
    print('[注意] TARGET_YEAR のデータが無いため全年で集計します')

s = tgt['n_obs_min'].dropna()
print('n_obs_min（各筆で最も観測が少なかった月の観測数）')
print(s.describe().to_string())
print()
print('しきい値を N にしたとき「判定不能」になる筆の割合:')
for th in range(0, 8):
    n = int((s < th).sum())
    print('  N_OBS_MIN = %d → 判定不能 %5d 筆 (%.1f%%)' % (th, n, 100 * n / len(s)))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].hist(s, bins=range(0, int(s.max()) + 2), edgecolor='white')
ax[0].set_title('n_obs_min の分布'); ax[0].set_xlabel('月あたり最小観測数'); ax[0].set_ylabel('筆数')
ax[1].hist(tgt['n_obs_total'].dropna(), bins=20, edgecolor='white')
ax[1].set_title('n_obs_total の分布'); ax[1].set_xlabel('4〜8月の合計観測数')
plt.tight_layout(); plt.show()

print('\n【判断の目安】観測が0の月がある筆は、その月について何も言えない。')
print('  5月（湛水）と8月（生育）は判定に直接使うため、この2か月が0の筆は')
print('  N_OBS_MIN の値によらず判定できない。上の表で 1 と 2 の差を見て決める。')

## Step4-5 / 4-6 — 判定と確信度

**判定は機械学習ではなくルールベース。**分岐は `config.py` の値で決まる。

| 分類 | 条件 |
|------|------|
| **判定不能** | 4月が欠測で湛水判定ができない、または `n_obs_min < N_OBS_MIN` |
| **水稲作付の可能性が高い** | 4月比 `VH_FLOOD_DROP_DB` dB 以上の落ち込みがある |
| **要確認** | 湛水シグナル× かつ NDVI 上昇○ |
| **水稲作付が確認できない** | 湛水シグナル× かつ NDVI 上昇× |

「湛水○ かつ 生育×」は `cfg.FLOOD_WITHOUT_GROWTH_CLASS` に従う（チーム決定 2026-09-05: 水稲作付）。
**要確認を `else` 分岐にしない。**残余クラスにすると判定の質と無関係に要確認率が膨らむ
（`analysis_doc`【要確認を残余クラスにしない】）。

In [ ]:
# ── Step4-5: 3段階分類 ＋ 判定不能 ──
CLS_RICE   = '水稲作付の可能性が高い'
CLS_CHECK  = '要確認'
CLS_NORICE = '水稲作付が確認できない'
CLS_NA     = '判定不能'

res = field_features.copy()

# --- 判定不能の条件 ---
undecidable = res['flood_detected'].isna()          # 4月欠測 → 湛水判定ができない
reason_na = pd.Series(np.where(undecidable, 'BASELINE_MISSING', ''), index=res.index)

if cfg.N_OBS_MIN is not None:
    low = res['n_obs_min'] < cfg.N_OBS_MIN
    reason_na = reason_na.mask(~undecidable & low, 'LOW_NOBS')
    undecidable = undecidable | low
else:
    print('[警告] cfg.N_OBS_MIN が None のため、観測数による「判定不能」を分離していません。')
    print('       上の分布セルで値を決め、config.py に記入してから再実行してください。')
    print('       現状の「判定不能」は4月欠測によるものだけです。')

flood  = res['flood_detected'].fillna(False).astype(bool)
growth = res['growth_detected'].fillna(False).astype(bool)

# 【重要】要確認は else 分岐ではなく明示条件（湛水× かつ 生育○）で入れる
res['zone'] = np.select(
    [undecidable,
     flood,                       # 湛水○ …「湛水○かつ生育×」もここへ（cfg で明示）
     ~flood &  growth,            # 湛水× かつ 生育○ → 要確認
     ~flood & ~growth],           # 湛水× かつ 生育× → 水稲作付が確認できない
    [CLS_NA, CLS_RICE, CLS_CHECK, CLS_NORICE],
    default=CLS_NA)

# 「湛水○ かつ 生育×」の行数を必ず表示する。暗黙に水稲作付へ入る組であり、
# 保全管理の湛水管理や一時的湛水がここに含まれうる（第2回レビュー 論点1）。
n_fng = int((flood & ~growth & ~undecidable).sum())
print('湛水○ かつ 生育× :', n_fng, '筆 →', cfg.FLOOD_WITHOUT_GROWTH_CLASS, 'に分類（チーム決定 2026-09-05）')

# --- Step4-6: 確信度（閾値からの距離）---
# predict_proba は使わない。閾値からどれだけ離れているかを 0〜1 に正規化する。
res['confidence'] = np.clip(
    (res['vh_drop_max'] - cfg.VH_FLOOD_DROP_DB).abs() / cfg.CONFIDENCE_SCALE_DB, 0, 1)
res.loc[undecidable, 'confidence'] = np.nan     # 判定不能に確信度は付けない

# --- 要確認の理由コード（★保留・cfg.REASON_CODES = None）---
# analysis_doc は「すべての要確認筆に理由コードを付与する」と規定しているが、
# コード体系は 2026-09-05 に保留とした。判定不能の理由のみ暫定で保持する。
res['reason'] = np.where(res['zone'] == CLS_NA, reason_na, '')
if cfg.REASON_CODES is None:
    print('[保留] 要確認の理由コードは未実装（config.py REASON_CODES = None）。')
    print('       要確認率 r を理由別に分解できない状態です。')

print()
print('=== 判定結果（%d年）===' % cfg.TARGET_YEAR)
tgt_res = res[res['year'] == cfg.TARGET_YEAR] if (res['year'] == cfg.TARGET_YEAR).any() else res
vc = tgt_res['zone'].value_counts()
for k in [CLS_RICE, CLS_CHECK, CLS_NORICE, CLS_NA]:
    n = int(vc.get(k, 0))
    print('  %-16s %5d 筆 (%.1f%%)' % (k, n, 100 * n / len(tgt_res)))

## 判別マップ

4色。**「要確認」と「判定不能」を必ず別色にする。**
観測が無くて判定できない筆と、観測があって判断がつかない筆は意味が正反対であるため
（`analysis_doc` Step4-6・8/20 レビュー 付録C ①）。

In [ ]:
# ── 判別マップ ──
year = cfg.TARGET_YEAR if (res['year'] == cfg.TARGET_YEAR).any() else res['year'].max()
if year != cfg.TARGET_YEAR:
    print('[注意] %d 年のデータが無いため %d 年で作図します' % (cfg.TARGET_YEAR, year))

gdf = fields_gdf.merge(res[res['year'] == year], on='field_id', how='inner')
print('結合できた筆数:', len(gdf), '/ 対象年:', year)
if len(gdf) == 0:
    _a, _b = set(fields_gdf['field_id']), set(res['field_id'])
    raise ValueError(
        'field_id が結合できません。\n'
        f'  fields.geojson  : {len(_a)} 筆  例 {list(_a)[:2]}\n'
        f'  field_features  : {len(_b)} 筆  例 {list(_b)[:2]}\n'
        f'  共通            : {len(_a & _b)} 筆\n'
        '件数が違う場合、fields.geojson が前回実行時の古いファイルである可能性が高い。\n'
        '04_field_pipeline_alpha.ipynb を最後まで再実行して両方を作り直してください。')

COLORS = {CLS_RICE: '#2e7d32',    # 緑
          CLS_CHECK: '#f9a825',   # 黄（要確認）
          CLS_NORICE: '#c62828',  # 赤
          CLS_NA: '#9e9e9e'}      # 灰（判定不能）

fig, ax = plt.subplots(figsize=(11, 7))
for z, c in COLORS.items():
    sub = gdf[gdf['zone'] == z]
    if len(sub):
        sub.plot(ax=ax, color=c, edgecolor='white', linewidth=0.3,
                 label='%s（%d筆）' % (z, len(sub)))
ax.set_title('相馬市 水稲作付の判別マップ（%d年・Phase 0 / 田 %d筆）' % (year, len(gdf)))
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(os.path.join(TEAM_DIR, 'fukushima_field_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('保存: fukushima_field_results.png')

## 現地確認の優先リスト

「水稲作付が確認できない」「要確認」の筆を**確信度の降順**で並べる
（`analysis_doc`【期待する出力】）。

In [ ]:
# ── 優先リスト ──
pri = (gdf[gdf['zone'].isin([CLS_NORICE, CLS_CHECK])]
       .sort_values('confidence', ascending=False)
       [['field_id', 'year', 'zone', 'confidence', 'vh_drop_max', 'vh_apr', 'vh_may',
         'ndvi_max', 'ndvi_aug', 'n_obs_min', 'reason']])
pri_path = os.path.join(TEAM_DIR, 'priority_check_list.csv')
pri.to_csv(pri_path, index=False)
print('優先確認リスト:', len(pri), '筆 →', pri_path)
pri.head(10)

## GIS で読める形に出力する

GeoPackage を主、Shapefile を従とする。Shapefile は列名10文字までの制限があり、
日本語の属性値は `encoding` を明示しないと文字化けする。

In [ ]:
# ── GIS 出力（必須項目6）──
out_cols = ['field_id', 'year', 'zone', 'confidence', 'flood_detected', 'growth_detected',
            'vh_apr', 'vh_may', 'vh_drop_max', 'ndvi_max', 'ndvi_aug',
            'n_obs_total', 'n_obs_min', 'reason', 'geometry']
out = gdf[[c for c in out_cols if c in gdf.columns]].copy()
# Shapefile は bool を持てないので文字列にする
for c in ['flood_detected', 'growth_detected']:
    if c in out.columns:
        out[c] = out[c].astype(str)

base = os.path.join(TEAM_DIR, 'fukushima_field_results')
out.to_file(base + '.gpkg', driver='GPKG')                 # GeoPackage（主）
out.to_file(base + '.shp', encoding='utf-8')               # Shapefile（従）
out.drop(columns='geometry').to_csv(base + '.csv', index=False)
print('保存:')
for ext in ['.gpkg', '.shp', '.csv']:
    print('  %s (%.1f KB)' % (base + ext, os.path.getsize(base + ext) / 1024))

# 読み直して壊れていないことを確認する
chk = gpd.read_file(base + '.gpkg')
print('\n読み直し確認: %d 筆 / CRS %s / クラス %s'
      % (len(chk), chk.crs, sorted(chk["zone"].unique())))

## サマリーと再現性ヘッダの記録

**費用対効果の試算は Phase 0 では出さない。**
新 ROI（約7.2 km²）は最小構成で一気通貫を確認するための設計であり、範囲の代表性は
要件ではない。ここで得た筆数・面積・要確認率 r を費用対効果に使うと、母集団と
報告単位が一致しない数字を自治体に示すことになる。
算定は Phase 2 の範囲（`ROI_PHASE2`・未確定）が決まってから行う。

In [ ]:
# ── サマリー（この実行の記録）──
print('=== 実行条件（再現性ヘッダ）===')
for k in ['ROI', 'TARGET_YEAR', 'MONTHS', 'CRS', 'SCALE', 'S1_ORBIT', 'S1_RELATIVE_ORBIT',
          'S2_CLEAR_PIXEL_RATIO', 'VH_FLOOD_DROP_DB', 'NDVI_GROWTH_START',
          'N_OBS_MIN', 'CONFIDENCE_SCALE_DB', 'CITY_CODES', 'LAND_TYPE', 'N_LIMIT']:
    print('  %-22s %s' % (k, getattr(cfg, k, '(未定義)')))

print('\n=== 結果（%d年・相馬市・田）===' % year)
print('  対象筆数        :', len(gdf))
vc = gdf['zone'].value_counts()
for k in [CLS_RICE, CLS_CHECK, CLS_NORICE, CLS_NA]:
    print('  %-16s %5d 筆 (%.1f%%)' % (k, int(vc.get(k, 0)), 100 * int(vc.get(k, 0)) / len(gdf)))
print('  確信度 中央値   : %.3f' % gdf['confidence'].median())

print('\n=== 未決事項（この実行に影響しているもの）===')
if cfg.N_OBS_MIN is None:
    print('  ★ N_OBS_MIN が未定。観測数による判定不能を分離していない')
if cfg.REASON_CODES is None:
    print('  ★ 要確認の理由コードが未実装。r を理由別に分解できない')
print('  ・確信度の正規化幅 %.1f dB は暫定値（未確認事項 #27）' % cfg.CONFIDENCE_SCALE_DB)
print('  ・費用対効果は Phase 0 では算定しない（母集団が報告単位と一致しないため）')

## 湛水検出の診断 — なぜ要確認が多いのか

Phase 0 の実行で**要確認が 53.4%** となった。対象は田であり多くは5月に湛水する水田のはずなので、
湛水シグナルが 24.3% しか出ないのは物理的におかしい。原因の候補は2つ。

1. **5 dB の閾値が実データに対して厳しすぎる**（未確認事項 #3）
2. **4月のベースラインが湛水で汚染されている**（未確認事項 #8）—
   4月の観測日は 04-06 / 04-18 / 04-30 と推定されているが実測未確認で、
   代かきが4月下旬に始まると 04-30 が既に湛水している可能性がある

下のセルで分布を見て切り分ける。**第2回レビュー（9/10）の持ち込み材料になる。**

In [ ]:
# ── 湛水検出の診断（未確認事項 #3 / #8 / #23）──
d = res[res['year'] == year].copy()

print('=== vh_drop_max（4月比の最大落ち込み・dB）===')
print(d['vh_drop_max'].describe().to_string())
print()
print('閾値ごとの「湛水あり」筆数:')
for th in [1, 2, 3, 4, 5, 6, 7, 8]:
    n = int((d['vh_drop_max'] >= th).sum())
    mark = '  ← 現在の閾値' if th == cfg.VH_FLOOD_DROP_DB else ''
    print('  %d dB 以上 → %5d 筆 (%5.1f%%)%s' % (th, n, 100 * n / len(d), mark))

print()
print('=== vh_drop_month（最大落ち込みが起きた月）===')
print('5月以外が多い場合、湛水以外の要因（豪雨・排水不良・スペックル）を拾っている疑い')
vc = d['vh_drop_month'].value_counts().sort_index()
for mo, n in vc.items():
    print('  %2d月: %5d 筆 (%.1f%%)' % (mo, n, 100 * n / len(d)))

print()
print('=== 月別 VH の中央値（dB）===')
for mo in cfg.MONTHS:
    col = f'vh_apr' if mo == 4 else (f'vh_may' if mo == 5 else (f'vh_aug' if mo == 8 else None))
    if col and col in d.columns:
        print('  %2d月: %.2f  (%s)' % (mo, d[col].median(), col))
print('  4月→5月の中央値の差: %.2f dB' % (d['vh_apr'].median() - d['vh_may'].median()))
print('  ※ この差が %.1f dB を大きく下回るなら、閾値かベースラインのどちらかに問題がある'
      % cfg.VH_FLOOD_DROP_DB)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].hist(d['vh_drop_max'].dropna(), bins=40, edgecolor='white')
ax[0].axvline(cfg.VH_FLOOD_DROP_DB, color='crimson', ls='--',
              label='閾値 %.1f dB' % cfg.VH_FLOOD_DROP_DB)
ax[0].set_xlabel('4月比の最大落ち込み (dB)'); ax[0].set_ylabel('筆数')
ax[0].set_title('湛水シグナルの分布'); ax[0].legend()

ax[1].hist(d['vh_apr'].dropna(), bins=40, alpha=0.6, label='4月（ベースライン）', edgecolor='white')
ax[1].hist(d['vh_may'].dropna(), bins=40, alpha=0.6, label='5月（湛水期）', edgecolor='white')
ax[1].set_xlabel('VH 後方散乱 (dB)'); ax[1].set_ylabel('筆数')
ax[1].set_title('4月と5月の VH 分布'); ax[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(TEAM_DIR, 'diag_flood_detection.png'), dpi=150, bbox_inches='tight')
plt.show()
print('保存: diag_flood_detection.png')

print()
print('【読み方】')
print(' ・5 dB の直下にピークがある     → 閾値の問題。値の見直しで大きく変わる')
print(' ・分布全体が 0〜2 dB に寄っている → 4月ベースラインの汚染、または湛水そのものが弱い')
print(' ・4月と5月の分布が重なっている   → ベースラインが既に湛水している疑い（#8）')

## 成果物をローカルにダウンロードする

判別マップ・診断グラフ・CSV・GIS ファイルをまとめて zip にして、ブラウザ経由で
ローカルに保存する。**Drive 上のファイルをそのまま右クリックで落としてもよいが、
枚数が多いのでまとめる。**

In [ ]:
# ── 成果物を zip にまとめてダウンロード ──
import zipfile, glob

TARGETS = [
    'fukushima_field_results.png',    # 判別マップ
    'diag_flood_detection.png',       # 湛水検出の診断グラフ
    'field_features.csv',             # 1筆×1年 の特徴量
    'priority_check_list.csv',        # 現地確認の優先リスト
    'fukushima_field_results.csv',    # 判定結果（属性のみ）
    'fukushima_field_results.gpkg',   # GeoPackage
    'fields.geojson',                 # 筆ジオメトリ
]
TARGETS += [os.path.basename(f) for f in
            glob.glob(os.path.join(TEAM_DIR, 'fukushima_field_results.*'))
            if f.rsplit('.', 1)[-1] in ('shp', 'shx', 'dbf', 'prj', 'cpg')]

zip_path = os.path.join(TEAM_DIR, f'team-alpha_phase0_{year}.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in dict.fromkeys(TARGETS):          # 重複を除く
        src = os.path.join(TEAM_DIR, name)
        if os.path.exists(src):
            z.write(src, name)
            print('  追加: %-34s %8.1f KB' % (name, os.path.getsize(src) / 1024))
        else:
            print('  （なし）', name)

print('\n作成: %s (%.1f MB)' % (zip_path, os.path.getsize(zip_path) / 1024 / 1024))

# Colab ならブラウザのダウンロードを起動する
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    print('ダウンロードを開始しました（ブラウザの許可が必要な場合があります）')
else:
    print('ローカル実行のため、上のパスから直接取得してください')

> **画像だけ欲しい場合**は次のように個別に落とせる。
>
> ```python
> from google.colab import files
> files.download(os.path.join(TEAM_DIR, 'fukushima_field_results.png'))
> ```
>
> ダウンロードがブラウザにブロックされる場合は、Colab 左のファイルペインから
> `drive/MyDrive/astrocamp-2026-sia/submissions/camp/team-alpha/` を開き、
> 対象ファイルの三点メニュー →「ダウンロード」でも取得できる。